---
title : "GWDI referentiestatistiek"
---

Dit notebook toont de workflow om `df_stats_minima` op te bouwen met de functies uit `gwdi_reference_stats`.

Voor GWDI wordt `df_stats_minima` **vooraf** berekend op basis van historische klimaatreeksen en opgeloste Pastas-modellen.

Die tabel wordt daarna gebruikt tijdens inference in `GwdiModel` voor de frequentie-naar-hoogte interpolatie per locatie.

Draai deze workflow opnieuw wanneer één van onderstaande inputs wijzigt:

- referentie-klimaatreeksen (neerslag/verdamping),
- locatieconfiguratie (`location`, `position`),
- Pastas-modelset,
- gekozen referentieperiode.

Het resultaat (bijvoorbeeld `df_stats_minima.csv`) hoort vervolgens als expliciete DataAdapter-input in GWDI-configuratie te staan.

Achtereenvolgens worden de volgende acties getoond:
1. Ruwe klimaatbestanden inlezen naar dataframes.
2. Klimaat voorbereiden met `prepare_reference_climate(...)` (naar m/dag en referentieperiode).
3. Locatie-info en Pastas-modellen laden en mappen op locatie.
4. `compute_df_stats_minima(...)` uitvoeren en optioneel wegschrijven.

In [ ]:
from pathlib import Path

import pandas as pd
import pastas as ps

from toolbox_continu_inzicht.base.adapters.input.pastas_models import (
    input_pastas_models,
)
from toolbox_continu_inzicht.gwdi.processing.gwdi_reference_stats import (
    compute_df_stats_minima,
    prepare_reference_climate,
)

Invoer:
- `df_precipitation_regions`: datetime-index + kolom `prec_R-R` (of andere regio-kolom), waarden in mm.
- `df_evaporation`: datetime-index + kolom `makkink`, waarden in mm.
- `info_locaties`: kolommen `location` en `position`.
- Pastas-modellen: keys in vorm `{location}_{position}_tarso`.

Uitvoer: 
- `df_stats_minima` met herhalingstijden als index (`1, 3, 10, 30, 100, 300, 1000`) en locaties als kolommen.

In [ ]:
# Gebruik lokale voorbeelddata in deze notebook-map.
data_dir = Path.cwd() / "data_sets" / "104.gwdi_reference_stats"
shared_dir = Path.cwd() / "data_sets" / "10.gwdi_simple_calculation"

path_precipitation = data_dir / "reference_precipitation_regions.csv"
path_evaporation = data_dir / "reference_evaporation.csv"
path_info = shared_dir / "gwdi_pastas_mapping.csv"
path_models = shared_dir / "pastas_models"

output_path = shared_dir / "df_stats_minima.csv"

In [ ]:
# 1) Laad ruwe klimaatinput als dataframes
df_precipitation_regions = pd.read_csv(
    path_precipitation,
    parse_dates=["time"],
    index_col="time",
)
df_evaporation = pd.read_csv(
    path_evaporation,
    parse_dates=["time"],
    index_col="time",
)

# 2) Maak klimaatreeksen in meter
prec100, evap100 = prepare_reference_climate(
    df_precipitation_regions=df_precipitation_regions,
    df_evaporation=df_evaporation,
    precipitation_column="prec_R-R",
    evaporation_column="makkink",
    start="1980-01-01",
    end="2014-12-31",
)

# 3) Laad locatie-info en Pastas-modellen
info_locaties = pd.read_csv(path_info)
info_locaties.index = info_locaties["location"].astype(str)
info_locaties["position"] = info_locaties["position"].astype(str)

dict_models_raw = input_pastas_models({"abs_path": path_models})
dict_models_by_location = {
    loc: dict_models_raw[f"{loc}_{info_locaties.loc[loc, 'position']}_tarso"]
    for loc in info_locaties.index
}

# 4) Bereken df_stats_minima
df_stats_minima = compute_df_stats_minima(
    info_locaties=info_locaties,
    dict_models=dict_models_by_location,
    prec100=prec100,
    evap100=evap100,
    ps_module=ps,
)

# Opslaan
df_stats_minima.to_csv(output_path)

df_stats_minima